# Train the basketball detector

Fine-tunes a small YOLOX-nano model (Apache-2.0, so no licence problems) to find the ball in
hoop-window pictures cut from your driveway footage, then scores it on footage it never saw.

**Before you start**
1. Runtime menu -> Change runtime type -> **T4 GPU**.
2. Put `dataset_v5.zip` in your Google Drive in a folder called `hardwood-lab`
   (so the path is `MyDrive/hardwood-lab/dataset_v5.zip`).
3. Run the cells in order, top to bottom. Training takes roughly 40-60 minutes.
4. When it finishes, `ball.onnx` and `results.txt` appear in `MyDrive/hardwood-lab/out5/`.
   Send `results.txt` back (copy and paste is fine).

Everything stays in your own Google account. Nothing is published.

The notebook stops at the first step that fails. If that happens, take a photo of the red error
text of that step and send it back. Do not keep clicking Run all.

In [ ]:
!nvidia-smi -L

## 1. Get the pictures from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/work
!unzip -q -o /content/drive/MyDrive/hardwood-lab/dataset_v5.zip -d /content/work
!ls /content/work/dataset

## 2. Install YOLOX

In [ ]:
%%bash
set -e
rm -rf /content/YOLOX
git clone --depth 1 https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

# YOLOX's own requirements.txt pins old versions (onnx-simplifier==0.4.10 among them) that
# cannot be built on Colab's current Python. These are the libraries it actually uses.
pip install -q loguru tabulate psutil tqdm thop pycocotools tensorboard packaging opencv-python-headless onnx onnxruntime

# YOLOX scores with an optional C++ speed-up that it compiles on the fly. That compile needs
# extra tools and fails on this very new PyTorch, so switch it off: YOLOX then falls back to the
# standard pycocotools scorer (slower, but it needs no compiling).
python - <<'EOF'
p = "/content/YOLOX/yolox/layers/__init__.py"
s = open(p).read()
assert "from .fast_coco_eval_api import COCOeval_opt" in s
s = s.replace("from .fast_coco_eval_api import COCOeval_opt", "raise ImportError('compiled scorer switched off')")
open(p, "w").write(s)
print("compiled scorer switched off")
EOF

cd /content/YOLOX
PYTHONPATH=/content/YOLOX python - <<'EOF'
import torch, torchvision, cv2, numpy, yolox
print("torch", torch.__version__, "| numpy", numpy.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime menu -> Change runtime type -> T4 GPU, then run again."
print("YOLOX imports fine")
EOF

## 3. Lay the pictures out the way YOLOX expects

In [ ]:
import os, shutil

D = '/content/work/dataset'
C = '/content/coco'
os.makedirs(f'{C}/annotations', exist_ok=True)
for split, name in [('train', 'train2017'), ('val', 'val2017')]:
    shutil.copytree(f'{D}/{split}', f'{C}/{name}', dirs_exist_ok=True)
    shutil.copy(f'{D}/{split}.json', f'{C}/annotations/instances_{name}.json')
print({s: len(os.listdir(f'{C}/{s}')) for s in ['train2017', 'val2017']})

## 4. Describe the model and the training

In [ ]:
os.makedirs('/content/YOLOX/exps/custom', exist_ok=True)
open('/content/YOLOX/exps/custom/ball_nano.py', 'w').write('''
import torch.nn as nn
from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super().__init__()
        # YOLOX-nano, one class (the ball), 416 x 416 pictures.
        self.depth = 0.33
        self.width = 0.25
        self.num_classes = 1
        self.input_size = (416, 416)
        self.test_size = (416, 416)
        self.random_size = (10, 20)
        self.mosaic_scale = (0.5, 1.5)
        self.mosaic_prob = 0.5
        self.enable_mixup = False
        self.data_dir = "/content/coco"
        self.data_num_workers = 2
        self.max_epoch = 40
        self.no_aug_epochs = 10
        self.warmup_epochs = 3
        self.eval_interval = 3
        self.exp_name = "ball_nano"

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if getattr(self, "model", None) is None:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(self.depth, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            head = YOLOXHead(self.num_classes, self.width, in_channels=in_channels, act=self.act, depthwise=True)
            self.model = YOLOX(backbone, head)
        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        self.model.train()
        return self.model
''')

## 5. Train (the long step)

In [ ]:
# Training runs in the background and this cell prints a progress line every 20 seconds.
# (Colab does not show the text of a shell step until the step ends, which made it look frozen.)
import os, subprocess, time

env = dict(os.environ, PYTHONPATH='/content/YOLOX',
           # newer PyTorch refuses some older checkpoint files unless told they are trusted (this one is from YOLOX itself)
           TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD='1')

subprocess.run(['wget', '-q', '-O', '/content/yolox_nano.pth',
                'https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth'], check=True)

log_path = '/content/train.log'
with open(log_path, 'w') as log:
    proc = subprocess.Popen(
        ['python', '-u', 'tools/train.py', '-f', 'exps/custom/ball_nano.py', '-d', '1', '-b', '32', '--fp16', '-o',
         '-c', '/content/yolox_nano.pth'],
        cwd='/content/YOLOX', env=env, stdout=log, stderr=subprocess.STDOUT)

started, last = time.time(), ''
while proc.poll() is None:
    time.sleep(20)
    lines = open(log_path).read().splitlines()
    useful = [l for l in lines if 'epoch:' in l or 'IoU=0.50 ' in l and 'Average Precision' in l]
    if useful and useful[-1] != last:
        last = useful[-1]
        print(f'{int((time.time() - started) / 60):>3} min  {last.split(" - ")[-1][:170]}')

if proc.returncode != 0:
    print(open(log_path).read()[-3500:])
    raise RuntimeError('Training stopped with an error. The end of its log is printed above; send a photo of it.')
if not os.path.exists('/content/YOLOX/YOLOX_outputs/ball_nano/best_ckpt.pth'):
    raise RuntimeError('Training ended but no best model was saved.')
print('TRAINING FINISHED: best model saved')

## 6. Turn the best model into the file the app uses

In [ ]:
# 1. Save the trained model to your Drive first, so it survives even if this session closes.
import os, shutil, subprocess, sys

os.makedirs('/content/drive/MyDrive/hardwood-lab/out5', exist_ok=True)
shutil.copy('/content/YOLOX/YOLOX_outputs/ball_nano/best_ckpt.pth', '/content/drive/MyDrive/hardwood-lab/out5/best_ckpt.pth')
print('trained model saved to Drive')

# 2. Turn it into the file the app uses. YOLOX's own export script calls a function that new PyTorch removed.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
sys.path.insert(0, '/content/YOLOX')
import torch
import torch.nn as nn
from yolox.exp import get_exp
from yolox.models.network_blocks import SiLU
from yolox.utils import replace_module

exp = get_exp('/content/YOLOX/exps/custom/ball_nano.py', None)
model = exp.get_model()
ckpt = torch.load('/content/YOLOX/YOLOX_outputs/ball_nano/best_ckpt.pth', map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model'])
model = replace_module(model, nn.SiLU, SiLU)
model.head.decode_in_inference = False
model = model.eval().cpu()
dummy = torch.randn(1, 3, 416, 416)

try:
    torch.onnx.export(model, dummy, '/content/ball.onnx', input_names=['images'], output_names=['output'],
                      opset_version=13, dynamo=False)
    print('exported with the classic exporter')
except Exception as e:
    print('classic exporter failed:', repr(e)[:300])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxscript'], check=True)
    program = torch.onnx.export(model, (dummy,), input_names=['images'], output_names=['output'], dynamo=True)
    program.save('/content/ball.onnx')
    print('exported with the new exporter')

# 3. Check the file works and has the shape the app expects.
import numpy as np
import onnxruntime as ort

session = ort.InferenceSession('/content/ball.onnx', providers=['CPUExecutionProvider'])
inp = session.get_inputs()[0]
out = session.run(None, {inp.name: (np.random.rand(1, 3, 416, 416) * 255).astype('float32')})[0]
print('input', inp.name, inp.shape, '| output', session.get_outputs()[0].name, out.shape)
assert out.shape == (1, 3549, 6), 'unexpected output shape ' + str(out.shape)
shutil.copy('/content/ball.onnx', '/content/drive/MyDrive/hardwood-lab/out5/ball.onnx')
print('BALL.ONNX READY and saved to Drive,', round(os.path.getsize('/content/ball.onnx') / 1e6, 1), 'MB')

## 7. Score it on footage it has never seen

The test pictures come from the last few minutes of the video, which training never touched.
Two numbers matter: how often the ball is found when it is really there, and how often it
invents a ball in a picture that has none.

In [ ]:
import json, cv2, numpy as np, onnxruntime as ort

session = ort.InferenceSession('/content/ball.onnx', providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name

grids, strides = [], []
for s in (8, 16, 32):
    n = 416 // s
    xv, yv = np.meshgrid(np.arange(n), np.arange(n))
    g = np.stack((xv, yv), 2).reshape(-1, 2)
    grids.append(g)
    strides.append(np.full((g.shape[0], 1), s))
grids, strides = np.concatenate(grids), np.concatenate(strides)

def detect(path, threshold):
    img = cv2.imread(path)  # BGR, 0-255, no normalising: what YOLOX expects
    x = img.astype(np.float32).transpose(2, 0, 1)[None]
    out = session.run(None, {input_name: x})[0][0].copy()
    out[:, :2] = (out[:, :2] + grids) * strides
    out[:, 2:4] = np.exp(out[:, 2:4]) * strides
    scores = out[:, 4] * out[:, 5]
    keep = scores >= threshold
    boxes, scores = out[keep], scores[keep]
    if len(boxes) == 0:
        return []
    xywh = [[float(b[0] - b[2] / 2), float(b[1] - b[3] / 2), float(b[2]), float(b[3])] for b in boxes]
    idx = cv2.dnn.NMSBoxes(xywh, [float(s) for s in scores], threshold, 0.45)
    return [(xywh[i], float(scores[i])) for i in np.array(idx).flatten()]

def iou(a, b):
    ax2, ay2, bx2, by2 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
    iw = max(0, min(ax2, bx2) - max(a[0], b[0]))
    ih = max(0, min(ay2, by2) - max(a[1], b[1]))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)

T = '/content/work/dataset'
coco = json.load(open(f'{T}/test.json'))
truth = {}
for a in coco['annotations']:
    truth.setdefault(a['image_id'], []).append(a['bbox'])

lines = []
lines.append(f"test pictures: {len(coco['images'])}, with a ball: {len(truth)}, without: {len(coco['images']) - len(truth)}")
lines.append("confidence | ball found when there (IoU>=0.3) | pictures with a made-up ball (of those with none)")
for thr in (0.05, 0.1, 0.25, 0.4, 0.6):
    found = total = fake = empty = 0
    for im in coco['images']:
        dets = detect(f"{T}/test/{im['file_name']}", thr)
        gts = truth.get(im['id'], [])
        if gts:
            for g in gts:
                total += 1
                found += any(iou(d[0], g) >= 0.3 for d in dets)
        else:
            empty += 1
            fake += len(dets) > 0
    lines.append(f"   {thr:>4}    |   {found}/{total} = {100 * found / max(1, total):.0f}%   |   {fake}/{empty} = {100 * fake / max(1, empty):.0f}%")
print("\n".join(lines))
open('/content/results.txt', 'w').write("\n".join(lines))

## 8. Save the results back to Drive

In [ ]:
%%bash
set -e
mkdir -p /content/drive/MyDrive/hardwood-lab/out5
cp /content/ball.onnx /content/results.txt /content/drive/MyDrive/hardwood-lab/out5/
cp /content/YOLOX/YOLOX_outputs/ball_nano/best_ckpt.pth /content/drive/MyDrive/hardwood-lab/out5/
ls -la /content/drive/MyDrive/hardwood-lab/out5